In [3]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import pandas as pd
import sys
import os
from pathlib import Path

In [4]:
sys.path.insert(1, os.path.abspath(".."))

In [5]:
df_all = pl.read_parquet("../data/preprocessed_all_data.parquet")

In [28]:
lf = df_all.lazy()

In [29]:

# The "Anchor" for your Rolling Window
expr_culture_orders = (
    # Look for the ORDER or PROCEDURE
    (pl.col("event_full_name").str.to_lowercase().str.contains("order")) &
    (pl.col("event_full_name").str.to_lowercase().str.contains("culture")) &
    
    # Filter for the actual REQUEST (Procedure), not the Result
    (pl.col("event_full_name").str.to_lowercase().str.contains("procedure"))
)


expr_antibiotics_gold_standard = (
    (pl.col("Event_Grouper") == "Antibiotics") & 
    
    # 1. INCLUSION: Must be a systemic route
    (pl.col("event_full_name").str.to_lowercase().str.contains(r"\biv\b|intravenous|infusion|injection|push|piggy back")) &
    
    # 2. EXCLUSION: The "Not Sepsis" Filter
    ~(pl.col("event_full_name").str.to_lowercase().str.contains(
        r"dialysis|"        # Maintenance fluids
        r"heparin|"         # Line flushes
        r"lock solution|"   # Catheter cleaning
        r"chemo|"           # Cancer treatment
        r"rubicin|"         # Specific chemo agents (Doxorubicin, etc.)
        r"epoch|"           # Chemo cocktail
        r"intravitreal|"    # Eye injections (Local)
        r"intrapleural|"    # Lung cavity (Local/Mechanical)
        r"alteplase|"       # Clot busters
        r"activase"         # Clot busters
    ))
)

def detect_sepsis_suspicion(lf: pl.LazyFrame, expr_antibiotics_gold_standard: pl.Expr,
                            expr_culture_orders: pl.Expr):
    lf_abx = lf.filter(
        expr_antibiotics_gold_standard
    ).select(
        "PAT_ENC_CSN_ID",
        pl.col("Event_DateTime").alias("ABX_Time")
    ).sort("ABX_Time")

    lf_culture = lf.filter(
        expr_culture_orders 
    ).select(
        "PAT_ENC_CSN_ID",
        pl.col("Event_DateTime").alias("culture_Time")
    ).sort("culture_Time")

    lf_suspected_backward = lf_abx.join_asof(
        lf_culture,
        left_on="ABX_Time",
        right_on="culture_Time",
        by="PAT_ENC_CSN_ID",
        strategy="backward",
        tolerance="24h"
    ).filter(pl.col("culture_Time").is_not_null())

    lf_suspected_forward = lf_abx.join_asof(
        lf_culture,
        left_on="ABX_Time",
        right_on="culture_Time",
        by="PAT_ENC_CSN_ID",
        strategy="forward",
        tolerance="72h"
    ).filter(pl.col("culture_Time").is_not_null())

    # Combine matches and pick the EARLIEST of the two times
    df_suspected_infection = (
        pl.concat([lf_suspected_backward, lf_suspected_forward])
        .unique()
        .with_columns(
            # Sepsis Time Zero = Min(Abx Time, Culture Time)
            pl.min_horizontal(["ABX_Time", "culture_Time"]).alias("Suspected_Infection_Time")
        )
        # Get the FIRST episode per encounter
        .group_by("PAT_ENC_CSN_ID")
        .agg(pl.col("Suspected_Infection_Time").min())
        .collect()
    )
    return df_suspected_infection, lf_suspected_backward, lf_suspected_forward

df_suspected_infection, lf_suspected_backward, lf_suspected_forward = detect_sepsis_suspicion(
    df_all.lazy(), expr_antibiotics_gold_standard, expr_culture_orders
)
print(df_suspected_infection.shape)

(3595, 2)


sys:1: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided


In [30]:
lf_suspected = pl.concat([lf_suspected_backward, lf_suspected_forward]).collect()

# SOFA score calculations

In [31]:
lf = lf.with_columns(
    pl.col("MEAS_VALUE")
    .str.extract(r"(-?\d+\.?\d*)", 1) # Regex to capture integer or float
    .cast(pl.Float64)
    .alias("val_numeric")
)

In [54]:
lf = lf.with_columns(
    pl.col("MEAS_VALUE")
    .str.extract(r"(-?\d+\.?\d*)", 1) # Regex to capture integer or float
    .cast(pl.Float64)
    .alias("val_numeric")
)

expr_bp = (
    pl.col("event_full_name").str.contains("BLOOD PRESSURE")
)

ALGO_bp_cond = (
    pl.col("event_full_name").str.contains("BLOOD PRESSURE")&
    pl.col("MEAS_VALUE").is_not_null()
)

lf = lf.join(
        lf.filter(
            ALGO_bp_cond
        ).with_columns(
            pl.col("MEAS_VALUE").str.split("/").list.get(0).cast(pl.Float64).alias("sys_bp"),
            pl.col("MEAS_VALUE").str.split("/").list.get(1).cast(pl.Float64).alias("dia_bp")
        ).select("PAT_ENC_CSN_ID", "Event_DateTime", "sys_bp", "dia_bp"),
        on=['PAT_ENC_CSN_ID', "Event_DateTime"],
        how='left'
).with_columns(
    MAP=((pl.col("sys_bp")+(2.0*pl.col("dia_bp")))/3.0).cast(pl.Float64)
)

In [55]:
lf.collect()

InvalidOperationError: division with 'String' datatypes is not allowed

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
LEFT JOIN:
LEFT PLAN ON: [col("PAT_ENC_CSN_ID"), col("Event_DateTime")]
   WITH_COLUMNS:
   [col("MEAS_VALUE").str.extract(["(-?\d+\.?\d*)"]).strict_cast(Float64).alias("val_numeric")] 
     WITH_COLUMNS:
     [col("MEAS_VALUE").str.extract(["(-?\d+\.?\d*)"]).strict_cast(Float64).alias("val_numeric")] 
      DF ["PAT_ENC_CSN_ID", "PAT_MRN_ID", "PAT_ID", "Ethnicity", ...]; PROJECT */25 COLUMNS
RIGHT PLAN ON: [col("PAT_ENC_CSN_ID"), col("Event_DateTime")]
  SELECT [col("PAT_ENC_CSN_ID"), col("Event_DateTime"), col("sys_bp"), col("dia_bp")]
     WITH_COLUMNS:
     [col("MEAS_VALUE").str.split(["/"]).list.get([dyn int: 0]).alias("sys_bp"), col("MEAS_VALUE").str.split(["/"]).list.get([dyn int: 1]).alias("dia_bp")] 
      FILTER col("MEAS_VALUE").is_not_null()
      FROM
        FILTER col("event_full_name").str.contains(["BLOOD PRESSURE"])
        FRO

In [15]:
lf.filter(
    (pl.col("event_full_name").str.to_lowercase().str.contains("creatin"))&
    (pl.col("event_full_name").str.to_lowercase().str.contains("order result"))
# ).select("event_full_name", "Type", "Event_Grouper", "MEAS_VALUE", "val_numeric", "Result_Flag").collect()
).collect()['event_full_name'].unique()

ComputeError: get index is out of bounds

In [14]:
# Platelet
expr_plat = (
    (pl.col('event_full_name').str.to_lowercase().str.contains("order result"))&
    (pl.col('event_full_name').str.to_lowercase().str.contains("platelet"))
)

expr_bilirubin = (
    (pl.col('event_full_name').str.to_lowercase().str.contains("order result"))&
    (pl.col('event_full_name').str.to_lowercase().str.contains("bili"))
)

expr_creatin = (
    (pl.col("event_full_name").str.to_lowercase().str.contains("creatin"))&
    (pl.col("event_full_name").str.to_lowercase().str.contains("order result"))
)

expr_bp = (
    pl.col("event_full_name").str.contains("BLOOD PRESSURE")
)
df_all.filter(expr_bp).select("event_full_name", "MEAS_VALUE").unique()

event_full_name,MEAS_VALUE
str,str
"""Flowsheet__BLOOD PRESSURE__Blo…","""156/107"""
"""Flowsheet__BLOOD PRESSURE__Blo…","""189/65"""
"""Flowsheet__BLOOD PRESSURE__Blo…","""105/30"""
"""Flowsheet__BLOOD PRESSURE__Blo…","""127/116"""
"""Flowsheet__BLOOD PRESSURE__Blo…","""140/50"""
…,…
"""Flowsheet__BLOOD PRESSURE__Blo…","""88/45"""
"""Flowsheet__BLOOD PRESSURE__Blo…","""105/56"""
"""Flowsheet__BLOOD PRESSURE__Blo…","""177/75"""


In [ ]:
expr_plat = (
    (pl.col('event_full_name').str.to_lowercase().str.contains("order result"))&
    (pl.col('event_full_name').str.to_lowercase().str.contains("platelet"))
)